#                                                          **<mark>PROJECT 1</mark>**

**<mark>case 1</mark>**

**CASE 1: Employee Performance Tracking****<mark>  
</mark>**

**Database:** `AdventureWorks2019`

**Problem Statement:**  
A company wants to evaluate employee performance using their pay history. Employees with a lower average pay rate may need support or training, while those with high pay are considered top performers. The company needs to:

- Identify employees with low average pay rates (below 20).
    
- Create a view that highlights top-performing employees with high rates (above 40).
    
- Provide a procedure to dynamically retrieve high-paid employees by passing a minimum rate threshold.
    

**SQL Features Used:**  
 Subquery  
 View  
 Stored Procedure  
 Joins

In [ ]:
-- CASE 1: Employee Performance Tracking (AdventureWorks2019)
USE AdventureWorks2019;
GO

-- Subquery: Employees with low avg pay (performance proxy)
SELECT e.BusinessEntityID, p.FirstName, p.LastName
FROM HumanResources.Employee e
JOIN Person.Person p ON e.BusinessEntityID = p.BusinessEntityID
WHERE e.BusinessEntityID IN (
    SELECT BusinessEntityID
    FROM HumanResources.EmployeePayHistory
    GROUP BY BusinessEntityID
    HAVING AVG(Rate) < 20
);
GO

-- View: Top-performing employees
DROP VIEW IF EXISTS vw_TopPerformers;
GO
CREATE VIEW vw_TopPerformers AS
SELECT e.BusinessEntityID, p.FirstName, p.LastName, eph.Rate
FROM HumanResources.Employee e
JOIN Person.Person p ON e.BusinessEntityID = p.BusinessEntityID
JOIN HumanResources.EmployeePayHistory eph ON e.BusinessEntityID = eph.BusinessEntityID
WHERE eph.Rate > 40;
GO

-- Procedure: Get employees with high pay
DROP PROCEDURE IF EXISTS sp_GetHighPayEmployees;
GO
CREATE PROCEDURE sp_GetHighPayEmployees @MinRate MONEY
AS
BEGIN
    SELECT e.BusinessEntityID, p.FirstName, p.LastName, eph.Rate
    FROM HumanResources.Employee e
    JOIN Person.Person p ON e.BusinessEntityID = p.BusinessEntityID
    JOIN HumanResources.EmployeePayHistory eph ON e.BusinessEntityID = eph.BusinessEntityID
    WHERE eph.Rate >= @MinRate;
END;
GO


**<mark>case 2</mark>**

**CASE 2: Fraudulent Transactions Detection****<mark>  
</mark>**

**Database:** `WideWorldImporters`

**Problem Statement:**  
The finance team at a bank suspects that some customers may be performing potentially fraudulent activities by making multiple high-value transactions in a short span of time. They want to:

- Identify individual transactions exceeding $10,000.
    
- Detect customers who repeatedly make transactions over $5,000.
    
- Calculate a **risk score** for each customer based on the number of high-value transactions.
    

**SQL Features Used:**  
 Basic Query  
 Common Table Expression (CTE)  
 Scalar Function  
 Aggregation  
 Filtering

In [ ]:
-- CASE 2: Fraudulent Transactions (WideWorldImporters)
USE WideWorldImporters;
GO

-- Query: High-value transactions
SELECT CustomerID, AmountExcludingTax, TransactionDate
FROM Sales.CustomerTransactions
WHERE AmountExcludingTax > 10000;
GO

-- CTE: Customers with repeated high-value txns
WITH FrequentSpenders AS (
    SELECT CustomerID, COUNT(*) AS TxnCount
    FROM Sales.CustomerTransactions
    WHERE AmountExcludingTax > 5000
    GROUP BY CustomerID
)
SELECT * FROM FrequentSpenders WHERE TxnCount > 3;
GO

-- Function: Risk score (number of high txns)
DROP FUNCTION IF EXISTS dbo.fn_RiskScore;
GO
CREATE FUNCTION dbo.fn_RiskScore (@CustomerID INT)
RETURNS INT
AS
BEGIN
    DECLARE @Score INT;
    SELECT @Score = COUNT(*) FROM Sales.CustomerTransactions
    WHERE CustomerID = @CustomerID AND AmountExcludingTax > 5000;
    RETURN @Score;
END;
GO


**<mark>case 3</mark>**

### **CASE 3: Online Shopping Order Delays**

**Database:** `TSQLV6`

**Problem Statement:**  
An e-commerce company wants to analyze delivery performance. They are particularly concerned about delays in order delivery and want to identify patterns that can help improve logistics. They aim to:

- Find orders that were delivered **later than the required date**.
    
- Identify **shippers** (delivery partners) who are frequently associated with delayed deliveries.
    
- Generate a **delay report** showing the number of days each delayed order was late.
    

**SQL Features Used:**  
 Basic Query  
 Subquery with `GROUP BY` and `HAVING`  
 Stored Procedure using `DATEDIFF`  
 Filtering and Aggregation

In [4]:
-- CASE 3: Order Delays (TSQLV6)
USE TSQLV6;
GO

-- Query: Orders delivered late
SELECT orderid, custid
FROM Sales.Orders
WHERE shippeddate > requireddate;
GO

-- Subquery: Delivery partners causing most delays
SELECT shipperid
FROM Sales.Orders
WHERE shippeddate > requireddate
GROUP BY shipperid
HAVING COUNT(*) > 2;
GO

-- Procedure: Delay report
DROP PROCEDURE IF EXISTS sp_DelayReport;
GO
CREATE PROCEDURE sp_DelayReport
AS
BEGIN
    SELECT orderid, custid,
           DATEDIFF(DAY, requireddate, shippeddate) AS DelayDays
    FROM Sales.Orders
    WHERE shippeddate > requireddate;
END;
GO


Commands completed successfully.

(37 rows affected)

(3 rows affected)

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.302

orderid,custid
10264,24
10271,75
10280,5
10302,76
10309,37
10320,87
10380,37
10423,31
10427,59
10433,60


shipperid
1
2
3


**<mark>case 4</mark>**

### **CASE 4: Customer Purchase Behavior Analysis**

**Database:** `StudentNorthwinds`

**Problem Statement:**  
Instead of traditional student grading, this case explores customer purchasing habits in the retail context. The business wants to:

- Identify customers who have spent less than $500 in total purchases.
    
- Group customers based on their overall spending into categories: **Low Value**, **Moderate**, and **High Value**.
    
- Retrieve a list of **loyal customers** who have placed more than 5 orders.
    

**SQL Features Used:**  
 Query with Join & Aggregation  
 Common Table Expression (CTE)  
 Stored Procedure  
 Grouping and Conditional Categorization

In [6]:
USE StudentNorthwinds;
GO

-- 1. Query: Customers with low total order value
SELECT c.CustomerId, c.CustomerCompanyName, SUM(od.UnitPrice * od.Quantity) AS TotalSpent
FROM Sales.Customer AS c
JOIN Sales.[Order] AS o ON c.CustomerId = o.CustomerId
JOIN Sales.OrderDetail AS od ON o.OrderId = od.OrderId
GROUP BY c.CustomerId, c.CustomerCompanyName
HAVING SUM(od.UnitPrice * od.Quantity) < 500;
GO

-- 2. CTE: Categorize customers by spending level
WITH CustomerSpending AS (
    SELECT c.CustomerId, c.CustomerCompanyName, SUM(od.UnitPrice * od.Quantity) AS TotalSpent
    FROM Sales.Customer AS c
    JOIN Sales.[Order] AS o ON c.CustomerId = o.CustomerId
    JOIN Sales.OrderDetail AS od ON o.OrderId = od.OrderId
    GROUP BY c.CustomerId, c.CustomerCompanyName
)
SELECT CustomerId, CustomerCompanyName,
       CASE
           WHEN TotalSpent >= 1000 THEN 'High Value'
           WHEN TotalSpent >= 500 THEN 'Moderate'
           ELSE 'Low Value'
       END AS Category
FROM CustomerSpending;
GO

-- 3. Stored Procedure: List loyal customers (more than 5 orders)
DROP PROCEDURE IF EXISTS sp_LoyalCustomers;
GO

CREATE PROCEDURE sp_LoyalCustomers
AS
BEGIN
    SELECT c.CustomerId, c.CustomerCompanyName, COUNT(*) AS TotalOrders
    FROM Sales.Customer AS c
    JOIN Sales.[Order] AS o ON c.CustomerId = o.CustomerId
    GROUP BY c.CustomerId, c.CustomerCompanyName
    HAVING COUNT(*) > 5;
END;
GO


Commands completed successfully.

(2 rows affected)

(89 rows affected)

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.093

CustomerId,CustomerCompanyName,TotalSpent
43,Customer UISOJ,357.00
13,Customer VMLOG,100.80


CustomerId,CustomerCompanyName,Category
23,Customer WVFAF,High Value
46,Customer XPNIK,High Value
69,Customer SIUIH,High Value
29,Customer MDLWA,Moderate
75,Customer XOJYP,High Value
15,Customer JUWXK,High Value
9,Customer RTXGC,High Value
89,Customer YBQTI,High Value
3,Customer KBUDE,High Value
52,Customer PZNLA,High Value


**<mark>case 5</mark>**

### **CASE 5: Inventory Management and Stock Depletion Forecast**

**Database:** `WideWorldImporters`

**Problem Statement:**  
A warehouse manager wants to proactively manage product inventory by identifying items that are low in stock and those frequently running out. The goal is to:

- List products with current stock levels below 50 units.
    
- Track frequently out-of-stock items using a view.
    
- Estimate how long current stock will last using a user-defined function that calculates depletion time based on historical usage.
    

**SQL Features Used:**  
Query with Join  
 View  
 Scalar Function  
 Aggregation and Filtering

In [7]:
-- CASE 5: Inventory Management (WideWorldImporters)
USE WideWorldImporters;
GO

-- 1. Query: Low stock products
SELECT h.StockItemID, i.StockItemName, h.QuantityOnHand
FROM Warehouse.StockItemHoldings AS h
JOIN Warehouse.StockItems AS i ON h.StockItemID = i.StockItemID
WHERE h.QuantityOnHand < 50;
GO

-- 2. View: Frequently out-of-stock items
DROP VIEW IF EXISTS vw_OutOfStock;
GO
CREATE VIEW vw_OutOfStock AS
SELECT StockItemID, COUNT(*) AS OutOfStockCount
FROM Warehouse.StockItemTransactions
WHERE Quantity < 1
GROUP BY StockItemID
HAVING COUNT(*) > 2;
GO

-- 3. Function: Estimate depletion time
DROP FUNCTION IF EXISTS dbo.fn_StockDepletion;
GO
CREATE FUNCTION dbo.fn_StockDepletion (@StockItemID INT)
RETURNS INT
AS
BEGIN
    DECLARE @AvgUsagePerDay INT;
    DECLARE @CurrentQty INT;

    -- Assume average daily usage = total quantity used / days (e.g., 30 days)
    SELECT @AvgUsagePerDay = ABS(SUM(Quantity)) / 30
    FROM Warehouse.StockItemTransactions
    WHERE StockItemID = @StockItemID AND Quantity < 0;

    SELECT @CurrentQty = QuantityOnHand
    FROM Warehouse.StockItemHoldings
    WHERE StockItemID = @StockItemID;

    RETURN CASE WHEN @AvgUsagePerDay = 0 THEN NULL ELSE @CurrentQty / @AvgUsagePerDay END;
END;
GO


Commands completed successfully.

(8 rows affected)

Commands completed successfully.

Commands completed successfully.

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.119

StockItemID,StockItemName,QuantityOnHand
77,"""The Gu"" red shirt XML tag t-shirt (White) XXS",27
78,"""The Gu"" red shirt XML tag t-shirt (White) XS",16
80,"""The Gu"" red shirt XML tag t-shirt (White) M",20
86,"""The Gu"" red shirt XML tag t-shirt (White) 5XL",3
95,"""The Gu"" red shirt XML tag t-shirt (Black) XL",48
98,"""The Gu"" red shirt XML tag t-shirt (Black) 4XL",25
184,Shipping carton (Brown) 305x305x305mm,38
204,Tape dispenser (Red),24


**<mark>case 6</mark>**

### **CASE 6: Sales Discount Analysis**

**Database:** `AdventureWorks2019`

**Problem Statement:**  
The sales department wants to analyze how discounts influence customer purchasing behavior. The team is particularly interested in customers who consistently receive high discounts and how discount thresholds affect sales performance. The tasks include:

- Identifying individual sales transactions where the **unit price discount is 20% or more**.
    
- Using a **CTE** to calculate the **average discount per customer**, and identifying those who average at least 15%.
    
- Using a **subquery** to find customers who received a high discount (20% or more) **in more than 3 orders**.
    
- Creating a **stored procedure** to dynamically retrieve transactions above a given discount threshold.
    

**SQL Features Used:**  
 Basic Query  
Common Table Expression (CTE)  
 Subquery  
 Stored Procedure  
Join and Aggregation

In [8]:
USE AdventureWorks2019;
GO

SELECT sod.SalesOrderID, sod.ProductID, sod.OrderQty, sod.UnitPrice, sod.UnitPriceDiscount
FROM Sales.SalesOrderDetail sod
WHERE sod.UnitPriceDiscount >= 0.20;
GO
WITH CustomerDiscounts AS (
  SELECT soh.CustomerID, AVG(sod.UnitPriceDiscount) AS AvgDiscount
  FROM Sales.SalesOrderHeader soh
  JOIN Sales.SalesOrderDetail sod ON soh.SalesOrderID = sod.SalesOrderID
  GROUP BY soh.CustomerID
)
SELECT * FROM CustomerDiscounts
WHERE AvgDiscount >= 0.15;
GO
SELECT CustomerID
FROM Sales.SalesOrderHeader
WHERE CustomerID IN (
  SELECT soh.CustomerID
  FROM Sales.SalesOrderHeader soh
  JOIN Sales.SalesOrderDetail sod ON soh.SalesOrderID = sod.SalesOrderID
  WHERE sod.UnitPriceDiscount >= 0.20
  GROUP BY soh.CustomerID
  HAVING COUNT(*) > 3
);
GO
DROP PROCEDURE IF EXISTS sp_HighDiscountOrders;
GO

CREATE PROCEDURE sp_HighDiscountOrders @MinDiscount DECIMAL(5, 2)
AS
BEGIN
    SELECT sod.SalesOrderID, sod.ProductID, sod.OrderQty, sod.UnitPrice, sod.UnitPriceDiscount
    FROM Sales.SalesOrderDetail sod
    WHERE sod.UnitPriceDiscount >= @MinDiscount;
END;
GO

-- Run it
EXEC sp_HighDiscountOrders @MinDiscount = 0.25;
GO


Commands completed successfully.

(598 rows affected)

(0 rows affected)

(442 rows affected)

Commands completed successfully.

...

Total execution time: 00:00:01.951

SalesOrderID,ProductID,OrderQty,UnitPrice,UnitPriceDiscount
51823,954,4,953.628,0.20
51823,957,21,953.628,0.20
51823,956,5,953.628,0.20
51826,955,6,953.628,0.20
51826,957,9,953.628,0.20
51826,954,4,953.628,0.20
51827,957,4,953.628,0.20
51827,955,1,953.628,0.20
51827,954,1,953.628,0.20
51829,956,1,953.628,0.20


CustomerID,AvgDiscount


CustomerID
29487
29487
29487
29487
29487
29487
29487
29487
29487
29487


SalesOrderID,ProductID,OrderQty,UnitPrice,UnitPriceDiscount
46323,771,1,849.9975,0.35
46323,772,1,849.9975,0.35
46323,773,1,849.9975,0.35
46323,774,1,849.9975,0.35
46327,776,2,843.7475,0.35
46327,774,3,849.9975,0.35
46327,778,2,843.7475,0.35
46327,775,5,843.7475,0.35
46327,771,2,849.9975,0.35
46330,773,2,849.9975,0.35


**<mark>case 7</mark>**

**Database:** `ContosoRetailDW`

**Problem Statement:**  
A streaming-like analytics platform wants to analyze product "view" behavior (simulated using online sales data). Customers represent users, and products represent streamed content. The team wants to:

- List the **most-watched products** based on total order frequency.
    
- Identify **inactive users** (customers who haven’t ordered anything in the last 3 months).
    
- Use a **function** to calculate the **average quantity (watch time)** per customer.
    
- Create a **view** to highlight **trending products** that are viewed more than 50 times.
    

**SQL Features Used:**  
 Query  
 Subquery  
 Scalar Function  
 View  
Aggregation & Filtering

In [9]:
USE ContosoRetailDW;
GO

-- 1. Most watched "movies" (products)
SELECT ProductKey, COUNT(*) AS TotalViews
FROM dbo.FactOnlineSales
GROUP BY ProductKey
ORDER BY TotalViews DESC;
GO

-- 2. Inactive users (no activity in last 3 months)
SELECT CustomerKey
FROM dbo.FactOnlineSales
GROUP BY CustomerKey
HAVING MAX(UpdateDate) < DATEADD(MONTH, -3, GETDATE());
GO

-- 3. Function: Average quantity ordered by a customer (like avg watch time)
DROP FUNCTION IF EXISTS dbo.fn_AvgQuantity;
GO

CREATE FUNCTION dbo.fn_AvgQuantity (@CustomerKey INT)
RETURNS FLOAT
AS
BEGIN
    DECLARE @Avg FLOAT;

    SELECT @Avg = AVG(SalesQuantity)
    FROM dbo.FactOnlineSales
    WHERE CustomerKey = @CustomerKey;

    RETURN @Avg;
END;
GO

-- 4. View: Trending products (watched > 50 times)
CREATE OR ALTER VIEW vw_TrendingProducts AS
SELECT ProductKey, COUNT(*) AS ViewCount
FROM dbo.FactOnlineSales
GROUP BY ProductKey
HAVING COUNT(*) > 50;
GO


Commands completed successfully.

(2516 rows affected)

Warning: Null value is eliminated by an aggregate or other SET operation.

(0 rows affected)

Commands completed successfully.

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:10.359

ProductKey,TotalViews
176,273615
153,273158
1052,116656
1293,112852
2517,105489
2506,96653
2491,96260
896,94639
2511,94080
864,93470


CustomerKey
